In [8]:
from mp_api.client import MPRester
from pathlib import Path
from pymatgen.io.vasp import Poscar
import json
import os
import re

MP_ID = 'Aiq22elbdTS1HTlv1DFWrDvdEPKWkLLL'

output_name = "input/candidates"
output_dir = Path(output_name)
output_dir.mkdir(parents=True, exist_ok=True)

c:\Users\mateu\Desktop\gnn-material-science\gnn-material-science\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
def generate_POSCAR(path_to_folder, doc):
    """Create POSCAR and metadata.json in formula/symmetry folder."""

    # Root folder
    if not os.path.exists(path_to_folder):
        os.mkdir(path_to_folder)

    # Formula folder
    path_to_folder = f"{path_to_folder}/{doc.formula_pretty}"
    if not os.path.exists(path_to_folder):
        os.mkdir(path_to_folder)

    # Symmetry folder
    symmetry_folder = re.sub('/', '-', doc.symmetry.symbol)
    path_to_folder = f"{path_to_folder}/{symmetry_folder}"
    if not os.path.exists(path_to_folder):
        os.mkdir(path_to_folder)

    # ---------- POSCAR ----------
    poscar_path = f"{path_to_folder}/POSCAR"
    poscar = Poscar(doc.structure)
    poscar.comment = f"{doc.formula_pretty} ({doc.material_id})"
    poscar.write_file(poscar_path)

    # ---------- metadata ----------
    metadata = {
        "material_id": str(doc.material_id),
        "formula": doc.formula_pretty,
        "symmetry": doc.symmetry.symbol,
        "energy_per_atom": doc.energy_per_atom,
        "band_gap": doc.band_gap
    }

    json_path = f"{path_to_folder}/metadata.json"
    with open(json_path, "w") as f:
        json.dump(metadata, f, indent=2)

In [10]:
elements = [
    'H','He','Li','Be','B','C','N','O','F','Ne','Na','Mg','Al','Si','P','S','Cl',
    'Ar','K','Ca','Ti','V','Cr','Mn','Fe','Co','Ni','Cu','Zn','Ga','Ge','As','Se','Br',
    'Kr','Sr','Y','Zr','Nb','Mo','Rh','Ag','Cd','In','Sn','Sb','Te','I','Ba','La','Ce',
    'Pr','Nd','Sm','Eu','Gd','Tb','Dy','Ho','Er','Yb','Lu','Hf','Ta','W','Hg','Pb','Bi',
    'At','Rn','Fr','Ra','Pa','U','Es','Fm','Md','No','Lr','Rf','Db','Sg','Bh','Hs','Mt',
    'Ds','Rg','Cn','Nh','Fl','Mc','Lv','Ts','Og','Cs','Pd','Au','Xe'
]

exclude_elements = [
    'Po','Ac','Tc','Cf','Bk','Cm','Pu','Am','Np','Pm',
    'Ir','Pt','Rb','Os','Ru','Tl','Sc','Re','Th','Tm'
]

fields = [
    "material_id",
    "formula_pretty",
    "structure",
    "symmetry",
    "energy_per_atom",
    "band_gap"
]

print("Querying Materials Project...")

with MPRester(MP_ID) as mpr:
    docs = mpr.materials.summary.search(
        exclude_elements=exclude_elements,
        fields=fields,
        all_fields=False
    )

print(f"Retrieved {len(docs)} materials")

Querying Materials Project...


Retrieving SummaryDoc documents: 100%|██████████| 131556/131556 [06:19<00:00, 347.04it/s]

Retrieved 131556 materials


In [12]:
for doc in docs:
    try:
        if doc.structure is None:
            continue

        generate_POSCAR(output_name, doc)

    except Exception as e:
        print("Skip", doc.material_id, e)

c:\Users\mateu\Desktop\gnn-material-science\gnn-material-science\.venv\Lib\site-packages\pymatgen\core\composition.py:1398: UserWarning: No Pauling electronegativity for He. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  return sorted(sym, key=lambda x: [float("inf") if math.isnan(e_neg := get_el_sp(x).X) else e_neg, x])
c:\Users\mateu\Desktop\gnn-material-science\gnn-material-science\.venv\Lib\site-packages\pymatgen\core\composition.py:1398: UserWarning: No Pauling electronegativity for Ar. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  return sorted(sym, key=lambda x: [float("inf") if math.isnan(e_neg := get_el_sp(x).X) else e_neg, x])
c:\Users\mateu\Desktop\gnn-material-science\gnn-material-science\.venv\Lib\site-packages\pymatgen\core\composition.py:1398: UserWarning: No Pauling electronegativity for Ne. Setting to NaN. This has no phys